# Initial Setting

In [1]:
import os, threadpoolctl

os.environ["OMP_NUM_THREADS"] = "1"     # OpenMP
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["BLIS_NUM_THREADS"] = "1"

threadpoolctl.threadpool_limits(1, "blas")
from threadpoolctl import threadpool_info
from threadpoolctl import threadpool_limits
threadpool_limits(1, "openmp")  # Limit OpenMP threads
threadpool_limits(1, "blas")    # Already done, but safe
threadpool_info()

[{'user_api': 'blas',
  'internal_api': 'mkl',
  'num_threads': 1,
  'prefix': 'libmkl_rt',
  'filepath': '/usr/local/lib/libmkl_rt.so.2',
  'version': '2025.3-Product',
  'threading_layer': 'gnu'},
 {'user_api': 'blas',
  'internal_api': 'mkl',
  'num_threads': 1,
  'prefix': 'libmkl_rt',
  'filepath': '/usr/local/lib/python3.11/dist-packages/mkl_fft.libs/libmkl_rt-089e6a60.so.2',
  'version': '2025.3-Product',
  'threading_layer': 'not specified'},
 {'user_api': 'openmp',
  'internal_api': 'openmp',
  'num_threads': 1,
  'prefix': 'libgomp',
  'filepath': '/usr/lib/x86_64-linux-gnu/libgomp.so.1.0.0',
  'version': None}]

In [2]:
!pip install implicit

class ImplicitALSWrapper:
    RECOMMENDER_NAME = "IALS_Wrapper_Model"
    
    def __init__(self, URM_train):
        self.URM_train = URM_train.tocsr()
        self.n_users, self.n_items = self.URM_train.shape
        self.items_to_ignore = np.array([], dtype=np.int64)
        self.model = None

    def fit(self, factors, regularization, alpha, iterations):
        self.model = AlternatingLeastSquares(
            factors=factors,
            regularization=regularization,
            alpha=alpha,
            iterations=iterations,
            use_cg=True,
            num_threads=4,
            dtype=np.float32,
        )
        self.model.fit(self.URM_train.astype(np.float32))

    def get_URM_train(self):
        return self.URM_train

    def set_items_to_ignore(self, items_to_ignore):
        self.items_to_ignore = np.array(items_to_ignore, dtype=np.int64)

    def reset_items_to_ignore(self):
        self.items_to_ignore = np.array([], dtype=np.int64)

    # ⭐ NEW: Needed for hybrid fusion
    def _compute_item_score(self, user_id_array, items_to_compute=None):
        """
        Calculates the score matrix for the given users.
        NOTE: items_to_compute=None is required to match the framework's call signature.
        """
        
        # Use the implicit library's user_factors and item_factors
        
        # Ensure user_id_array is a flat list/array of indices
        user_factors = self.model.user_factors[user_id_array, :]
        
        if items_to_compute is None:
            item_factors = self.model.item_factors
        else:
            item_factors = self.model.item_factors[items_to_compute, :]
        
        # Score calculation: User_factors * Item_factors.T
        # Note: The result should be shape (N_users, N_items)
        scores = user_factors.dot(item_factors.T)
        
        # If items_to_compute was specified, the scores are only for those items
        if items_to_compute is not None:
            # The framework expects the scores for the full item set, 
            # so you need to map back to the full matrix if required by the recommender base class.
            
            # Since the BaseRecommender often handles the mapping, 
            # the most common fix is to ensure the output is a NumPy array:
            return np.asarray(scores)
        
        return scores
    def recommend(self, user_id_array, remove_seen_flag=True, cutoff=None,
                  remove_top_pop_flag=False, items_to_compute=None,
                  remove_custom_items_flag=False, return_scores=False):

        user_id_array = np.array(user_id_array, dtype=np.int64)
        if cutoff is None:
            cutoff = self.n_items

        scores_batch = self._compute_item_score(user_id_array)

        if remove_custom_items_flag and self.items_to_ignore.size > 0:
            scores_batch[:, self.items_to_ignore] = -np.inf

        if remove_seen_flag:
            for idx, user_id in enumerate(user_id_array):
                seen_items = self.URM_train[user_id].indices
                scores_batch[idx, seen_items] = -np.inf

        recommended_items_batch_list = []

        for row_idx in range(scores_batch.shape[0]):
            user_scores = scores_batch[row_idx]
            k = min(cutoff, len(user_scores))
            top_k_idx = np.argpartition(-user_scores, k - 1)[:k]
            top_k_idx = top_k_idx[np.argsort(-user_scores[top_k_idx])]
            recommended_items_batch_list.append(top_k_idx)

        if return_scores:
            return recommended_items_batch_list, scores_batch
        else:
            return recommended_items_batch_list


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 60.8 MB/s eta 0:00:0000:0100:01


In [3]:
from kaggle_secrets import UserSecretsClient
secret_label = "RecSysToken"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/JieCver1/RecSysChanllengePolimi25-26.git

Cloning into 'RecSysChanllengePolimi25-26'...
remote: Enumerating objects: 191, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 191 (delta 40), reused 182 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (191/191), 10.82 MiB | 22.11 MiB/s, done.
Resolving deltas: 100% (40/40), done.


In [4]:
! pip install PyGithub requests
#! pip install cython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 8.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 42.3 MB/s eta 0:00:00


In [5]:
%cd /kaggle/working/RecSysChanllengePolimi25-26

/kaggle/working/RecSysChanllengePolimi25-26


In [6]:
import warnings
import os

# Suppress Python warnings
warnings.filterwarnings("ignore")

# Suppress GCC / NumPy deprecated API warnings
os.environ["CFLAGS"] = "-Wno-unused-function -Wno-uninitialized -Wno-cpp"
os.environ["NPY_NO_DEPRECATED_API"] = "NPY_1_7_API_VERSION"

!python run_compile_all_cython.py


run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorization_Cython_Epoch.pyx... 
/usr/local/lib/python3.11/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /kaggle/working/RecSysChanllengePolimi25-26/Recommenders/MatrixFactorization/Cython/MatrixFactorization_Cython_Epoch.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
Compiling [1/11]: MatrixFactorization_Cython_Epoch.pyx... PASS

Compiling [2/11]: MatrixFactorizationImpressions_Cython_Epoch.pyx... 
/usr/local/lib/python3.11/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /kaggle/working/RecSysChanllengePolimi25-26/Recommenders/Matr

# Import Libraries

In [7]:
import numpy as np
import pandas as pd
import scipy.sparse as sps
import sys
import optuna

In [8]:
from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender, SLIMElasticNetRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from implicit.als import AlternatingLeastSquares
from Recommenders.NonPersonalizedRecommender import TopPop 
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender

np.random.seed(42)

# Prepare dataset

In [9]:
df_train = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv')
df_target_users = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv')

df_train.rename(columns={'row':'user_id','col':'item_id'}, inplace=True)
df_train['data'] = 1

user_ids = df_train['user_id'].unique()
item_ids = df_train['item_id'].unique()

user_to_index = {u: i for i, u in enumerate(user_ids)}
index_to_user = {i: u for u, i in user_to_index.items()}
item_to_index = {i: j for j, i in enumerate(item_ids)}
index_to_item = {j: i for i, j in item_to_index.items()}

# Build URM_all
URM_all = sps.csr_matrix(
    (df_train['data'],
     (df_train['user_id'].map(user_to_index),
      df_train['item_id'].map(item_to_index))),
    shape=(len(user_ids), len(item_ids))
)

print("URM_all Shape:", URM_all.shape)

URM_all Shape: (27095, 6969)


In [10]:
import scipy.sparse as sps

URM_train = sps.load_npz("/kaggle/input/datasettuning/URM_train.npz")
URM_validation = sps.load_npz("/kaggle/input/datasettuning/URM_validation.npz")

In [11]:
'''from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

# Second split: this is ONLY for meta-learning (XGBoost training)
URM_meta_train, URM_meta_valid = split_train_in_two_percentage_global_sample(
    URM_train,
    train_percentage=0.8  # 80% of URM_train → 64% global
)

print("URM_meta_train:", URM_meta_train.shape)
print("URM_meta_valid:", URM_meta_valid.shape)'''

'from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample\n\n# Second split: this is ONLY for meta-learning (XGBoost training)\nURM_meta_train, URM_meta_valid = split_train_in_two_percentage_global_sample(\n    URM_train,\n    train_percentage=0.8  # 80% of URM_train → 64% global\n)\n\nprint("URM_meta_train:", URM_meta_train.shape)\nprint("URM_meta_valid:", URM_meta_valid.shape)'

In [12]:
evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[20])

# TEST combination

In [32]:
n_users, n_items = URM_train.shape
feature_cols = [
    "TopPop", "SLIM", "RP3", "EASER", "IALS", "ItemKNN", "UserKNN",
    "score_mean", "score_std", "log_item_pop", "log_user_prof",
    "SLIMxRP3", "SLIMxEASER", "RP3xIALS", "KNNxEASER", "ItemKNNxUserKNN", 
    "rank_TopPop", "rank_SLIM", "rank_RP3", "rank_EASER", 
    "rank_IALS", "rank_ItemKNN", "rank_UserKNN"
]

In [33]:
def train_base_models(URM):
    print("Training base models on URM with shape:", URM.shape)

    # 1) SLIM (Candidate Generator)
    slim = MultiThreadSLIM_SLIMElasticNetRecommender(URM)
    slim.fit(l1_ratio=0.0148035519731873, alpha=0.004908131512399756, topK=920, positive_only=True, workers=4)

    # 2) RP3
    rp3 = RP3betaRecommender(URM)
    rp3.fit(topK=53, alpha=1.4261147318499703, beta=0.33422325175727774, normalize_similarity=True)

    # 3) ItemKNN
    item_knn = ItemKNNCFRecommender(URM)
    item_knn.fit(topK=1089, shrink=107, similarity='cosine', feature_weighting='BM25', normalize=True)

    # 4) EASER
    easer = EASE_R_Recommender(URM)
    easer.fit(l2_norm=418.6313286374365, topK=1501)

    # 5) IALS (NOTE: Requires ImplicitALSWrapper definition)
    ials = ImplicitALSWrapper(URM) 
    ials.fit(factors=112, regularization=0.08220470709553462, alpha=15.001334807331522, iterations=46)
    
    # 6) TopPop
    top_pop = TopPop(URM)
    top_pop.fit()

    # 7) UserKNN (NEW MODEL)
    user_knn = UserKNNCFRecommender(URM)
    user_knn.fit(topK=300, shrink=100, similarity='cosine', normalize=True, feature_weighting='TF-IDF')

    base_models = {
        "TopPop": top_pop,
        "SLIM": slim,
        "RP3": rp3,
        "EASER": easer,
        "IALS": ials,
        "ItemKNN": item_knn,
        "UserKNN": user_knn, # <-- ADDED
    }

    return base_models

In [34]:
def get_multi_source_candidates(user_id, base_models, max_candidates=700):
    
    # Define tiered cutoffs based on model strength/purpose
    C_SLIM = 400            # High cutoff for best core candidate model
    C_DIVERSITY = 150       # Moderate cutoff for RP3beta, ItemKNN, UserKNN
    C_MF = 250              # Mid-level cutoff for IALS (MF models benefit from depth)

    cand = set()

    # --- 1. Candidate Filter (SLIM) ---
    slim = base_models["SLIM"]
    cand |= set(slim.recommend(user_id, cutoff=C_SLIM))

    # --- 2. Diversity/Graph Models (RP3, ItemKNN, UserKNN) ---
    rp3  = base_models["RP3"]
    knn  = base_models["ItemKNN"]
    user_knn = base_models["UserKNN"]
    
    cand |= set(rp3.recommend(user_id, cutoff=C_DIVERSITY))
    cand |= set(knn.recommend(user_id, cutoff=C_DIVERSITY))
    cand |= set(user_knn.recommend(user_id, cutoff=C_DIVERSITY))

    # --- 3. MF Model (IALS) ---
    ials = base_models["IALS"]
    ials_recs = ials.recommend([user_id], cutoff=C_MF)[0]
    cand |= set(ials_recs)
    
    # Total pool size will be the union of 400 + 150*3 + 250 = 1100, 
    # then capped at max_candidates (e.g., 700)

    # Trim candidates
    cand = list(cand)
    if len(cand) > max_candidates:
        cand = cand[:max_candidates]

    return np.array(cand, dtype=np.int64)

In [35]:
from Recommenders.BaseRecommender import BaseRecommender

class XGBoostReranker(BaseRecommender):

    def __init__(self, URM, base_models, XGB_model, feature_cols,
                 cutoff_candidates=700):
        super(XGBoostReranker, self).__init__(URM)

        self.base_models = base_models
        self.XGB_model = XGB_model
        self.feature_cols = feature_cols
        self.cutoff_candidates = cutoff_candidates

        # Features computed on the URM used for training the base models (URM_train)
        self.item_popularity = np.ediff1d(sps.csc_matrix(URM).indptr)
        self.user_profile_len = np.ediff1d(sps.csr_matrix(URM).indptr)

    def _get_candidates(self, user_id):
        # Assumes get_multi_source_candidates is globally defined and updated 
        # to include SLIM, RP3, IALS, ItemKNN, and UserKNN, and uses the correct cutoffs.
        return get_multi_source_candidates(
            user_id,
            self.base_models,
            max_candidates=self.cutoff_candidates # Pass the final desired pool size
        )

    def recommend(
        self,
        user_id_array,
        cutoff=20,
        remove_seen_flag=True,
        remove_top_pop_flag=False,
        remove_custom_items_flag=False,
        items_to_compute=None,
        return_scores=False
    ):

        n_items = self.n_items
        recommended_items_batch = []
        scores_batch = [] 
        
        # Define score_cols here for local use within the recommend method
        # This list must match the models used for score_mean/score_std in training
        score_cols = ["SLIM", "RP3", "EASER", "IALS", "ItemKNN", "UserKNN"] 

        for user_id in user_id_array:

            item_list = self._get_candidates(user_id)
            if len(item_list) == 0:
                recommended_items_batch.append(np.full(cutoff, -1, dtype=np.int64))
                # Append dummy score vector to satisfy EvaluatorHoldout (expected 2 return values)
                scores_batch.append(np.full(n_items, -np.inf, dtype=np.float32)) 
                continue

            item_list = np.array(item_list, dtype=np.int64)

            # 1) Build DataFrame X_user for feature calculation
            X_user = pd.DataFrame(index=item_list)

            # 2) Scores and Ranks calculation (Iterates over all base models)
            for rec_label, rec_model in self.base_models.items():
                # Compute scores for ALL items, then extract only the candidates' scores
                all_scores = rec_model._compute_item_score([user_id])[0]
                scores = all_scores[item_list]
                X_user[rec_label] = scores

                # Calculate ranks (0 = best)
                order = np.argsort(-scores)
                ranks = np.empty_like(order)
                ranks[order] = np.arange(len(order))
                X_user[f"rank_{rec_label}"] = ranks

            # 3) Popularity and Profile features
            X_user["item_popularity"] = self.item_popularity[item_list]
            X_user["user_profile_len"] = self.user_profile_len[user_id] # user_profile_len is constant for this user

            # 4) Engineered features (must match features used in training!)
            X_user["score_mean"] = X_user[score_cols].mean(axis=1)
            X_user["score_std"]  = X_user[score_cols].std(axis=1).fillna(0)
            
            X_user["log_item_pop"]  = np.log1p(X_user["item_popularity"])
            X_user["log_user_prof"] = np.log1p(X_user["user_profile_len"])
            
            # Interaction features
            X_user["SLIMxRP3"]   = X_user["SLIM"] * X_user["RP3"]
            X_user["SLIMxEASER"] = X_user["SLIM"] * X_user["EASER"]
            X_user["RP3xIALS"]   = X_user["RP3"]  * X_user["IALS"]
            X_user["KNNxEASER"]  = X_user["ItemKNN"] * X_user["EASER"]
            X_user["ItemKNNxUserKNN"] = X_user["ItemKNN"] * X_user["UserKNN"] # Added interaction

            # 5) Final feature preparation
            # Ensure all columns exist and are ordered correctly before prediction
            for col in self.feature_cols:
                if col not in X_user.columns:
                    X_user[col] = 0.0

            X_user = X_user[self.feature_cols].astype(np.float32)

            # 6) Prediction (Reranking Score)
            y_pred = self.XGB_model.predict(X_user.values)

            # 7) Rerank candidates based on XGBoost score
            scores_all = np.full(n_items, -np.inf, dtype=np.float32)
            scores_all[item_list] = y_pred # Map XGBoost score back to the item's position

            order = np.argsort(-y_pred)
            top_items = item_list[order[:cutoff]]

            # Padding if not enough candidates were generated
            if len(top_items) < cutoff:
                pad = np.full(cutoff - len(top_items), -1, dtype=np.int64)
                top_items = np.concatenate([top_items, pad])

            recommended_items_batch.append(top_items)
            scores_batch.append(scores_all) # Store the full score vector for the Evaluator

        recommended_items_batch = np.vstack(recommended_items_batch)
        scores_batch = np.vstack(scores_batch) 

        # Return both items and scores (required by EvaluatorHoldout)
        return recommended_items_batch, scores_batch

In [30]:
final_base_models = train_base_models(URM_train)

Training base models on URM with shape: (27095, 6969)


100%|█████████▉| 6968/6969 [03:10<00:00, 36.56it/s]


RP3betaRecommender: Similarity column 6969 (100.0%), 2213.45 column/sec. Elapsed time 3.15 sec
Similarity column 6969 (100.0%), 2376.91 column/sec. Elapsed time 2.93 sec
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 17.50 sec


  0%|          | 0/46 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 872.85 column/sec. Elapsed time 31.04 sec


In [36]:
from tqdm import tqdm
print("\n--- Generating Aligned L2R Training Data ---")

URM_train_for_features = URM_train       
URM_labels_for_XGB = URM_validation      
CANDIDATE_CUTOFF_FINAL=700
# 3.A Candidate & Label Generation
rows_final = []
for user_id in tqdm(range(n_users), desc="3A. Candidate Generation"):
    item_list = get_multi_source_candidates(user_id, final_base_models,
                                            max_candidates=CANDIDATE_CUTOFF_FINAL)
    if len(item_list) == 0:
        continue
    rows_final.extend([(user_id, int(i)) for i in item_list])

training_dataframe_final = pd.DataFrame(rows_final, columns=["UserID", "ItemID"])

URM_labels_coo_final = sps.coo_matrix(URM_labels_for_XGB)
correct_recommendations_final = pd.DataFrame({
    "UserID": URM_labels_coo_final.row, "ItemID": URM_labels_coo_final.col})

training_dataframe_final = pd.merge(
    training_dataframe_final,
    correct_recommendations_final,
    on=["UserID", "ItemID"],
    how="left",
    indicator="Exist",)

training_dataframe_final["Label"] = training_dataframe_final["Exist"] == "both"
training_dataframe_final.drop(columns=["Exist"], inplace=True)


--- Generating Aligned L2R Training Data ---


3A. Candidate Generation: 100%|██████████| 27095/27095 [01:31<00:00, 297.33it/s]


In [37]:
item_popularity_final = np.ediff1d(sps.csc_matrix(URM_train_for_features).indptr)
user_profile_len_final = np.ediff1d(sps.csr_matrix(URM_train_for_features).indptr)

training_dataframe_final["item_popularity"] = item_popularity_final[training_dataframe_final["ItemID"].values.astype(int)]
training_dataframe_final["user_profile_len"] = user_profile_len_final[training_dataframe_final["UserID"].values.astype(int)]
training_dataframe_final = training_dataframe_final.sort_values(["UserID", "ItemID"]).reset_index(drop=True)

# Calculate Scores and Ranks for all models (including UserKNN)
for user_id in tqdm(range(n_users), desc="3B. Feature Calculation"):
    mask = (training_dataframe_final["UserID"] == user_id)
    idx  = training_dataframe_final.index[mask]
    if len(idx) == 0: continue
    item_list = training_dataframe_final.loc[idx, "ItemID"].values.astype(int)

    for rec_label, rec_model in final_base_models.items():
        all_scores = rec_model._compute_item_score([user_id])[0]
        scores = all_scores[item_list]
        training_dataframe_final.loc[idx, rec_label] = scores
        
        order = np.argsort(-scores)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(len(order))
        training_dataframe_final.loc[idx, f"rank_{rec_label}"] = ranks


3B. Feature Calculation: 100%|██████████| 27095/27095 [12:45<00:00, 35.38it/s]


In [38]:
# 4. Engineered Features
score_cols = ["SLIM", "RP3", "EASER", "IALS", "ItemKNN", "UserKNN"] # <-- NEW MODEL
training_dataframe_final["score_mean"] = training_dataframe_final[score_cols].mean(axis=1).astype(np.float32)
training_dataframe_final["score_std"]  = training_dataframe_final[score_cols].std(axis=1).fillna(0).astype(np.float32)
training_dataframe_final["log_item_pop"]  = np.log1p(training_dataframe_final["item_popularity"].astype(np.float32))
training_dataframe_final["log_user_prof"] = np.log1p(training_dataframe_final["user_profile_len"].astype(np.float32))

# Interaction Features (Update these to include UserKNN interactions if desired)
training_dataframe_final["SLIMxRP3"]   = training_dataframe_final["SLIM"] * training_dataframe_final["RP3"]
training_dataframe_final["SLIMxEASER"] = training_dataframe_final["SLIM"] * training_dataframe_final["EASER"]
training_dataframe_final["RP3xIALS"]   = training_dataframe_final["RP3"]  * training_dataframe_final["IALS"]
training_dataframe_final["KNNxEASER"]  = training_dataframe_final["ItemKNN"] * training_dataframe_final["EASER"]
training_dataframe_final["ItemKNNxUserKNN"] = training_dataframe_final["ItemKNN"] * training_dataframe_final["UserKNN"]

# Prepare XGBoost Tensors
# Ensure feature_cols includes "UserKNN" and "rank_UserKNN"
X_train_final = training_dataframe_final[feature_cols].astype(np.float32)
y_train_final = training_dataframe_final["Label"].astype(np.float32)
groups_final = training_dataframe_final.groupby("UserID").size().values

In [39]:
print("\n--- 5. Training Final XGBoost Ranker ---")
from xgboost import XGBRanker

XGB_model_final = XGBRanker(
    objective="rank:pairwise", 
    n_estimators=350,       # Reduced from 600 (to stop early and prevent noise memorization)
    learning_rate=0.03,
    max_depth=5,            # Reduced from 6 (Crucial complexity reduction)
    min_child_weight=0.1,
    colsample_bytree=0.7,
    subsample=0.85,
    reg_alpha=10.0,         # Increased L1 regularization (Fights overfitting)
    reg_lambda=5.0,         # Increased L2 regularization (Fights overfitting)
    tree_method="gpu_hist", 
    device="cuda", 
    booster="gbtree",
    random_state=42, 
    verbosity=1,
)

XGB_model_final.fit(
    X_train_final,
    y_train_final,
    group=groups_final,
    eval_set=[(X_train_final, y_train_final)],
    eval_group=[groups_final],
    eval_metric="ndcg@20",
    verbose=True,)


--- 5. Training Final XGBoost Ranker ---
[0]	validation_0-ndcg@20:0.33512
[1]	validation_0-ndcg@20:0.33959
[2]	validation_0-ndcg@20:0.34344
[3]	validation_0-ndcg@20:0.34472
[4]	validation_0-ndcg@20:0.34523
[5]	validation_0-ndcg@20:0.34465
[6]	validation_0-ndcg@20:0.34530
[7]	validation_0-ndcg@20:0.34542
[8]	validation_0-ndcg@20:0.34547
[9]	validation_0-ndcg@20:0.34557
[10]	validation_0-ndcg@20:0.34543
[11]	validation_0-ndcg@20:0.34558
[12]	validation_0-ndcg@20:0.34577
[13]	validation_0-ndcg@20:0.34594
[14]	validation_0-ndcg@20:0.34607
[15]	validation_0-ndcg@20:0.34619
[16]	validation_0-ndcg@20:0.34623
[17]	validation_0-ndcg@20:0.34612
[18]	validation_0-ndcg@20:0.34611
[19]	validation_0-ndcg@20:0.34638
[20]	validation_0-ndcg@20:0.34640
[21]	validation_0-ndcg@20:0.34641
[22]	validation_0-ndcg@20:0.34646
[23]	validation_0-ndcg@20:0.34642
[24]	validation_0-ndcg@20:0.34653
[25]	validation_0-ndcg@20:0.34651
[26]	validation_0-ndcg@20:0.34654
[27]	validation_0-ndcg@20:0.34663
[28]	validation_

XGBRanker(base_score=None, booster='gbtree', callbacks=None,
          colsample_bylevel=None, colsample_bynode=None, colsample_bytree=0.7,
          device='cuda', early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
          importance_type=None, interaction_constraints=None,
          learning_rate=0.03, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
          max_leaves=None, min_child_weight=0.1, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=350,
          n_jobs=None, num_parallel_tree=None, objective='rank:pairwise', ...)

In [40]:
print("\n--- 5. Evaluating Final Aligned Reranker ---")

reranker_final_eval = XGBoostReranker(
    URM_train,             # Matrix for Pop/Len features (URM_train)
    final_base_models,     # Base models trained on URM_train
    XGB_model_final,       # The L2R model trained on aligned features
    feature_cols,
    cutoff_candidates=700)

evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[20])
results_final, results_string_final = evaluator_validation.evaluateRecommender(reranker_final_eval)

print("✅ Final Evaluation with Aligned XGBoost Reranker:")
print(results_string_final)


--- 5. Evaluating Final Aligned Reranker ---
EvaluatorHoldout: Processed 15000 (55.4%) in 5.06 min. Users per second: 49
EvaluatorHoldout: Processed 27095 (100.0%) in 9.11 min. Users per second: 50
✅ Final Evaluation with Aligned XGBoost Reranker:
CUTOFF: 20 - PRECISION: 0.2301587, PRECISION_RECALL_MIN_DEN: 0.3508650, RECALL: 0.2961359, MAP: 0.1079063, MAP_MIN_DEN: 0.1602224, MRR: 0.5524462, NDCG: 0.3303803, F1: 0.2590118, HIT_RATE: 0.9755674, ARHR_ALL_HITS: 1.0282432, NOVELTY: 0.0286858, AVERAGE_POPULARITY: 0.3115756, DIVERSITY_MEAN_INTER_LIST: 0.9346353, DIVERSITY_HERFINDAHL: 0.9967300, COVERAGE_ITEM: 0.4099584, COVERAGE_ITEM_HIT: 0.3060697, ITEMS_IN_GT: 0.9985651, COVERAGE_USER: 1.0000000, COVERAGE_USER_HIT: 0.9755674, USERS_IN_GT: 1.0000000, DIVERSITY_GINI: 0.0753198, SHANNON_ENTROPY: 9.3129654, RATIO_DIVERSITY_HERFINDAHL: 0.9975016, RATIO_DIVERSITY_GINI: 0.2783113, RATIO_SHANNON_ENTROPY: 0.8305885, RATIO_AVERAGE_POPULARITY: 1.4726927, RATIO_NOVELTY: 0.1984397, 



## Train final hybrid model

In [42]:
final_base_models = train_base_models(URM_all)

Training base models on URM with shape: (27095, 6969)



100%|█████████▉| 6968/6969 [04:03<00:00, 28.62it/s]


RP3betaRecommender: Similarity column 6969 (100.0%), 1821.48 column/sec. Elapsed time 3.83 sec
Similarity column 6969 (100.0%), 1769.46 column/sec. Elapsed time 3.94 sec
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 18.21 sec


  0%|          | 0/46 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 690.76 column/sec. Elapsed time 39.22 sec


NameError: name 'XGB_model' is not defined

In [45]:
reranker_final = XGBoostReranker(
    URM_all,
    final_base_models,  
    XGB_model_final,
    feature_cols,
    cutoff_candidates=700)

In [47]:
print("\n🚀 Generating FINAL submission using XGBoost reranker...")

# Load target users
target_users_df = pd.read_csv(
    "/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv"
)
target_user_ids = target_users_df["user_id"].tolist()

# Ensure URM_all is used for popularity fallback, as required for final submission
item_pop = np.ediff1d(sps.csc_matrix(URM_all).indptr)
popular_items = np.argsort(-item_pop)

submission = []

for user_id in target_user_ids:

    if user_id in user_to_index:   # Known user
        u = user_to_index[user_id]

        # Use the finalized reranker (must be trained on anti-overfit params!)
        # The cutoff_candidates size must be 700 here
        recommendations_batch = reranker_final.recommend([u], cutoff=20, remove_seen_flag=True)
        recommendations = recommendations_batch[0]   # This is the 1D array of item IDs

        # FIX: Robustly convert NumPy array elements to Python scalars (integers)
        # This handles cases where NumPy arrays of size 1 might be present.
        item_id_list = [index_to_item[int(i)] for i in recommendations.flatten()]
        
        # Filter out padding values (-1) that might still be present from the Reranker's padding
        item_id_list = [item for item in item_id_list if item != -1]


    else:   # Cold user fallback
        # Ensure only known item IDs are used for fallback
        item_id_list = [index_to_item[int(i)] for i in popular_items[:20]]

    # Guarantee exactly 20 items (for submission format)
    item_id_list = item_id_list[:20]
    if len(item_id_list) < 20:
        # Use -1 as a placeholder/padding if less than 20 items are found (though uncommon with cutoff=20)
        item_id_list += [-1] * (20 - len(item_id_list))

    submission.append({
        "user_id": int(user_id),
        "item_list": " ".join(map(str, item_id_list))
    })

# Save CSV
submission_df = pd.DataFrame(submission)
submission_df.to_csv("submission_XGBoost_hybrid_v5.csv", index=False)

print("🎉 Final submission_XGBoost.csv CREATED!")
print(submission_df.head())


🚀 Generating FINAL submission using XGBoost reranker...
🎉 Final submission_XGBoost.csv CREATED!
   user_id                                          item_list
0        0  2530 6411 2392 5532 4486 828 3049 2399 2168 27...
1        1  5630 4936 6810 3460 2509 2907 5223 2628 2295 8...
2        2  5640 2557 4264 5323 5390 5369 1539 5723 2529 4...
3        3  3713 1458 4161 6329 6739 2873 336 1554 2653 72...
4        4  4373 3512 2808 4835 2167 2472 5369 6684 5356 1...
